# TAHAPAN ANALISIS SENTIMEN

# IMPORT LIBRARY

In [ ]:
import tweepy
import csv
import pandas as pd
import numpy as np
import nltk
import string, re
import matplotlib.pyplot as plt
import nltk.corpus

from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory, StopWordRemover, ArrayDictionary
from Sastrawi.Dictionary.ArrayDictionary import ArrayDictionary
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from textblob import TextBlob
from nltk.tokenize import word_tokenize

from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report
from sklearn.metrics import f1_score, recall_score, precision_score, accuracy_score
from sklearn.metrics import roc_curve, auc, roc_auc_score
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer


# CRAWLING

In [ ]:
#KONEKSI ANTARA TWITTER API DAN PROGRAM UNTUK CRAWLING
api_key = 'INSERT API KEY'
api_key_secret = 'INSERT API KEY SECRET'
access_token = 'INSERT ACCESS TOKEN'
access_token_secret = 'INSERT ACCESS TOKEN SECRET'

auth = tweepy.OAuthHandler(api_key, api_key_secret)
auth.set_access_token(access_token, access_token_secret)
api = tweepy.API(auth, wait_on_rate_limit=True)
search_key = 'IKN baru'


In [ ]:
csvFile = open('crawling_'+search_key+'.csv', 'a+', newline='', encoding='utf-8')
csvWriter = csv.writer(csvFile)
u = []
c = []
t = []

for tweet in tweepy.Cursor(api.search_tweets, q=search_key+' -filter:retweets',
                           tweet_mode='extended', lang='id', count=420).items(420):
    print(tweet.user.name, tweet.created_at, tweet.full_text)
    u.append(tweet.user.name)
    c.append(tweet.created_at)
    t.append(tweet.full_text.encode("utf-8"))
    csvWriter.writerow([tweet.user.name, tweet.created_at, tweet.full_text])

In [ ]:
df = pd.read_csv('crawling_'+search_key+'.csv', names=['username', 'tanggal', 'isi_tweet'], sep=',')
df

# LABELING

In [ ]:
df = pd.read_csv('labeling IKN Baru.csv', sep=',')
df

In [ ]:
df['label'].value_counts()

# PREPROCESSING

In [ ]:
# CASE FOLDING
def cleansing(data):
    
    # lower text
    data = data.lower()

    # hapus mention
    data = ' '.join(re.sub('([@#]+[A-Za-z0-9]+)|([@_#]+[A-Za-z0-9]+)|(\w+:\/\/\S+)',' ', data).split())
                  
    # hapus hastag/#tagger
    data = re.sub(r'#([^\s]+)', ' ', data)
    
    # hapus html
    data = re.sub(r'http\S+', ' ', data)
    
    # hapus ASCII dan unicode
    data = data.encode('ascii', 'ignore').decode('utf-8')
    data = re.sub(r'[^\x00-\x7f]',r' ', data)
         
    # hapus angka
    data = re.sub(r'\d+', ' ', data)
        
    # hapus single char
    data = re.sub(r'\b[a-zA-Z]\b', ' ', data)
    
    # hapus remove tab, new line, dan back slice
    data = data.replace('\\t',' ').replace('\\n',' ').replace('\\u',' ').replace('\\',' ')

    # hapus punctuation/tanda baca
    data = data.translate(str.maketrans(' ',' ', string.punctuation))

    return data

In [ ]:
# jalankan cleansing data
df['case_folding'] = df['isi_tweet'].apply(cleansing)
df.head()

In [ ]:
#Tahap Stopword
stop_factory = StopWordRemoverFactory().get_stop_words()
more_stopword = ['ad','ada','adalah','adanya','adapun','adlh','agak','agaknya','agar','akan','akankah','agr','ah','aj','aja',
                 'ajaa','aje','akhir','akhiri','akhirnya','aku','akulah','ama','amat','amatlah','amp','an','anda','andalah',
                 'andai','antar','antara','antaranya','apa','apaa','apaan','apaapa','apabila','apakah','apalagi','apatah',
                 'artinya','asal','asalkan','atas','atau','ataukah','ataupun','atw','awal','awalnya',
                 'bagai','bagaikan','bagaimana','bagaimanakah','bagaimanapun','bagi','bagian','bahkan','bahwa','bahwasanya',
                 'baik','baiknya','bakal','bakalan','balik','banyak','bapak','bawah','beberapa','begini','beginian',
                 'beginikah','beginilah','begitu','begitukah','begitulah','begitupun','bekerja','belakang','belakangan',
                 'belum','belumlah','benarkah','benarlah','berada','berakhir','berakhirlah','berakhirnya','berapa','berapakah',
                 'berapalah','berapapun','berarti','berawal','berbagai','berdatangan','beri','berikan','berikut','berikutnya',
                 'berjumlah','berkali','berkata','berkehendak','berkeinginan','berkenaan','berlainan','berlalu','berlangsung',
                 'berlebihan','bermacam','bermacam','bermaksud','bermula','bersama','bersama','bersiap','bertanya','berturut',
                 'bertutur','berujar','berupa','besar','betul','betulkah','biasa','biasanya','bila','bilakah','bisa','bisakah',
                 'boleh','bolehkah','bolehlah','buat','bukan','bukankah','bukanlah','bukannya','bulan','bung','bbrp','bbrpa',
                 'bo','bro','brp','brarti','bs','btl','bu',
                 'cara','caranya','cc','cm','cukup','cukupkah','cukuplah','cuma',
                 'dah','dahulu','dahuluada','dalam','dan','dann','dapat','dari','darimana','daripada','datang','deh','dekat',
                 'demi','demikian','demikianlah','dengan','depan','dg','dgn','di','dia','dialah','diakhiri','diakhirinya',
                 'dialah','diantara','diantaranya','diberi','diberikan','diberikannya','dibuat','dibuatnya','didapat',
                 'didatangkan','digunakan','diibaratkan','diibaratkannya','diingat','diingatkan','diinginkan','dijawab',
                 'dijelaskan','dijelaskannya','dikarenakan','dikatakan','dikatakannya','dikerjakan','diketahui','diketahuinya',
                 'dikira','dilakukan','dilalui','dilihat','dimaksud','dimaksudkan','dimaksudkannya','dimaksudnya','diminta',
                 'dimintai','dimisalkan','dimulai','dimulailah','dimulainya','dimungkinkan','dini','dipastikan','diperbua',
                 'diperbuatnya','dipergunakan','diperkirakan','diperlihatkan','diperlukan','diperlukannya','dipersoalkan',
                 'dipertanyakan','dipunyai','diri','dirinya','disampaikan','disebut','disebutkan','disebutkannya','disini',
                 'disinilah','ditambahkan','ditandaskan','ditanya','ditanyai','ditanyakan','ditegaskan','ditujukan','ditunjuk',
                 'ditunjuki','ditunjukkan','ditunjukkannya','ditunjuknya','dituturkan','dituturkannya','diucapkan',
                 'diucapkannya','diungkapkan','dl','dlu','dong','dpat','dpt','dr','dri','drimana','drimna','dripada','drpd',
                 'drpda','dua','dulu','duh','duuh','duuuh',
                 'eh','ehh','entar','entr','empat','enggak','enggaknya','entah','entahlah',
                 'gimana','gmana','gimna','gmna','gmn','gua','gue','guna','gunakan','gw','gwa','gwe',
                 'ha','hadap','hai','hal','halo','haloo','hallo','helo','hello','helloo','hampir','hanya','hanyalah','hari',
                 'harus','haruslah','harusnya','hendak','hendaklah','hendaknya','hingga',
                 'ia','ialah','ibarat','ibaratkan','ibaratnya','ibu','ii','ikut','ingat','ingin','inginkah','inginkan','ini',
                 'inikah','inilah','itu','itukah','itulah','iy','iya','iyaa','iyaa',
                 'jadi','jd','jdi','jadilah','jadinya','jangan','jangankan','janganlah','jauh','jawab','jawaban',
                 'jawabnya','jelas','jelaskan','jelaslah','jelasnya','jika','jikalau','juga','jumat','jumlah','jumlahnya',
                 'justru',
                 'kala','kalau','kalaulah','kalaupun','kali','kalian','kalo','kata','kau','klo','klau','klu','kami','kamilah',
                 'kamis','kamu','kamulah','kan','kann','kannn','kapan','kapankah','kapanpun','karena','karna','krna','krn',
                 'krena','karenanya','kasus','kata','katakan','katakanlah','katanya','kau','ko','ke','keadaan','kebetulan',
                 'kecil','kedua','keduanya','keinginan','kek','kelamaan','kelihatan','kelihatannya','kelima','keluar',
                 'kembali','kemudian','kemungkinan','kemungkinannya','kenapa','kepada','kepadanya','keq','koq','kesampaian',
                 'keseluruhan','keseluruhannya','keterlaluan','ketika','khususnya','kini','kinilah','kira','kiranya','kita',
                 'kitalah','km','kmu','kmuu','kmudian','knapa','knp','knpa','kok','kurang','kurangnya',
                 'lagi','lagia','lah','lh','lain','lainnya','lalu','lama','lamanya','lanjut','lanjutnya','lebih','lewat',
                 'lima','loh','lo','loe','lue','lu','luar',
                 'macam','maka','makanya','makin','malah','malahan','mampu','mampukah','mana','manakala','manalagi','masa',
                 'masalah','masalahnya','masih','masihkah','masing','mata','mau','maupun','melainkan','melakukan','melalui',
                 'melihat','melihatnya','memang','memastikan','memberi','memberikan','membuat','memerlukan','memihak',
                 'meminta','memintakan','memisalkan','memperbuat','mempergunakan','memperkirakan','memperlihatkan',
                 'mempersiapkan','mempersoalkan','mempertanyakan','mempunyai','memulai','memungkinkan','menaiki',
                 'menambahkan','menandaskan','menanti','menantikan','menanya','menanyai','menanyakan','mendapat','mendapatkan',
                 'mendatang','mendatangi','mendatangkan','menegaskan','mengakhiri','mengapa','mengatakan','mengatakannya',
                 'mengenai','mengerjakan','mengetahui','menggunakan','menghendaki','mengibaratkan','mengibaratkannya',
                 'mengingat','mengingatkan','menginginkan','mengira','mengucapkan','mengucapkannya','mengungkapkan','menjadi',
                 'menjawab','menjelaskan','menuju','menunjuk','menunjuki','menunjukkan','menunjuknya','menurut','menuturkan',
                 'menyampaikan','menyangkut','menyatakan','menyebutkan','menyeluruh','menyiapkan','merasa','mereka',
                 'merekalah','merupakan','meski','meskipun','meyakini','meyakinkan','minggu','minta','mirip','misal','misalkan',
                 'misalnya','mreka','mrk','mrka','mula','mulai','mulailah','mulanya','mungkin','mungkinkah','mw',
                 'nah','nahh','naik','namun','nang','nanti','nantinya','ni','nih','nihh','nihhh','ntar','ntr','nya','nyaa',
                 'nek','neng','nnti','nanti','nyaris','nyatanya',
                 'oh','ohh','ohhh','ooh','olah','oleh','olehnya',
                 'pada','padahal','padanya','pak','paling','panjang','pantas','para','pasti','pastilah','penting',
                 'pentingnya','per','percuma','perlu','perlukah','perlunya','pernah','persoalan','pertama','pertanyaan',
                 'pertanyakan','pdhal','pdhl','pula','pun','pihak','pihaknya','pukul','pula','pun','punya',
                 'rabu','rasa','rasanya','rata','rupanya',
                 'saat','saatnya','sabtu','saja','sajalah','saling','sama','sama','sambil','sampai','sampaikan','sana',
                 'sangat','sangatlah','sapa','satu','saya','sayalah','sb','sbb','sbagai','sbgai','sbg','sbelum','sblum',
                 'sblm','sbelumnya','sblumnya','sblmnya','sbelumny','sblumny','sblmny','sdang','sdg','se','seandai',
                 'seandainya','sebab','sebabnya','sebagai','sebagaimana','sebagainya','sebagian','sebaik','sebaiknya',
                 'sebaliknya','sebanyak','sebegini','sebegitu','sebelum','sebelumnya','sebenarnya','seberapa','sebesar',
                 'sebetulnya','sebisanya','sebuah','sebut','sebutlah','sebutnya','secara','secukupnya','sedang','sedangkan',
                 'sedemikian','sedikit','sedikitnya','seenaknya','segala','segalanya','segera','seharusnya','sehingga',
                 'seingat','sejak','sejauh','sejenak','sejumlah','sekadar','sekadarnya','sekali','sekalian','sekaligus',
                 'sekalipun','sekarang','sekarang','sekecil','seketika','sekiranya','sekitar','sekitarnya','sekurang',
                 'sekurangnya','sela','selain','selaku','selalu','selama','selama','selamanya','selanjutnya','selasa','seluruh',
                 'seluruhnya','semacam','semakin','semampu','semampunya','semasa','semasih','semata','semaunya','sementara',
                 'semisal','semisalnya','sempat','semua','semuanya','semula','sendiri','sendirian','sendirinya','senin','seolah',
                 'seorang','sepanjang','sepantasnya','sepantasnyalah','seperlunya','seperti','sepertinya','sepihak','sering',
                 'seringnya','serius','seriuskah','serta','serupa','sesaat','sesama','sesampai','sesegera','sesekali','seseorang','sesuatu',
                 'sesuatunya','sesudah','sesudahnya','setelah','setempat','setengah','seterusnya','setiap','setiba',
                 'setibanya','setidak','setidaknya','setinggi','seusai','sewaktu','shingga','shg','si','siap','siapa',
                 'siapakah','siapapun','sini','sinilah','sn','sna','sni','soal','soalnya','spa','sprti','suatu','sudah',
                 'sudahkah','sudahlah','supaya','sya','sy',
                 'tadi','tadinya','tahu','tahun','tak','tambah','tambahnya','tampak','tampaknya','tandas','tandasnya',
                 'tanpa','tanya','tanyakan','tanyanya','tapi','tegas','tegasnya','telah','tempat','tengah','tentang',
                 'tentu','tentulah','tentunya','tepat','terakhir','terasa','terbanyak','terdahulu','terdapat','terdiri',
                 'terhadap','terhadapnya','teringat','teringat','terjadi','terjadilah','terjadinya','terkira','terlalu',
                 'terlebih','terlihat','termasuk','ternyata','tersampaikan','tersebut','tersebutlah','tertentu','tertuju',
                 'terus','terutama','tetap','tetapi','tiap','tiba','tidak','tidakkah','tidaklah','tidaknya','tiga',
                 'tinggi','tju','toh','tu','tuu','tuh','tuhh','tuhhh','tuju','tuk','tunjuk','turut','tutur','tuturnya',
                 'ucap','ucapnya','ujar','ujarnya','umum','umumnya','ungkap','ungkapnya','untuk','usah','usai','uwaw',
                 'waduh','wah','wahai','waktu','waktunya','walau','walaupun','waw','waaw','waww','wkk','wkwk','wkwkwk',
                 'wkwkwkchuakzz','wong',
                 'ya','yaa','yah','yahh','yaah','yaitu','yak','yakin','yakni','yan','yang','yng','ybs','ye','yg','yo',
                 'yok','yokk'
                ]

# merge stopword
data = stop_factory + more_stopword

dictionary = ArrayDictionary(data)
stopword = StopWordRemover(dictionary)

review = []
for index, row in df.iterrows():
    review.append(stopword.remove(row['case_folding']))
    
# jalankan proses stopword removal
df['stopword'] = review
df.head()

In [ ]:
# Stemmer
factory = StemmerFactory()
stemmer = factory.create_stemmer()

# jalankan proses stemmer
review = []
for index, row in df.iterrows():
    review.append(stemmer.stem(row['stopword']))

df['stemming']=review
df.head()

In [ ]:
#Tokenization
def token(data):
    return word_tokenize(data)
# jalankan proses tokenizing
review = []
for index, row in df.iterrows():
    review.append(token(row['stemming']))

df['token']=review
df.head()

In [ ]:
# Menyimpan data clean
df = df.to_csv('labeling IKN BARU CLEAN.csv',sep=',', index=False)
df_clean = pd.read_csv('labeling IKN BARU CLEAN.csv', sep=',')
df_clean

# SPLIT DATA

In [ ]:
# Deklarasi Variabel
X = df_clean['token']
y = df_clean['label']

# Membagi data testing dan training
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, stratify=y, random_state=30)

In [ ]:
print(y_train)
df_yTrain = pd.DataFrame(data = y_train)
df_yTrain = df_yTrain.to_csv('y_train.csv',sep=',')

# EKSTRAKSI FITUR

In [ ]:
# Vectorizer Object
tfidf = TfidfVectorizer()

# TF-IDF
X_train = tfidf.fit_transform(X_train)
X_test = tfidf.transform(X_test)

# Get Feature Names
tfidfTokens = tfidf.get_feature_names()

In [ ]:
# Menampilkan Nilai Vectorizer
df_tfidf = pd.DataFrame(data = X_train.toarray(),columns = tfidfTokens)

# Menyimpan data Vectorizer
df_tfidf = df_tfidf.to_csv('tfidfVec.csv',sep=',', index=False)

df_tfidf = pd.read_csv('tfidfVec.csv',sep=',')

In [ ]:
# Menampilkan Nilai TF-IDF Vectorizer
df_tfidf

In [ ]:
print(tfidfTokens)

In [ ]:
nb = MultinomialNB().fit(X_train, y_train)

# Menentukan hasil prediksi dari x_test
pred = nb.predict(X_test)

# Menentukan probabilitas hasil prediksi
prob = nb.predict_proba(X_test)[:,1]

In [ ]:
#print (y_test) sebagai label aktual
df_yTest = pd.DataFrame(data = y_test)
df_yTest

In [ ]:
#perbandingan label aktual dengan label prediksi
df_yTest['Prediksi']=pred
df_yTest = df_yTest.to_csv('y_testPred.csv',sep=',')
df_yTest = pd.read_csv('y_testPred.csv', sep=',')
df_yTest

In [ ]:
print('MultinomialNB Accuracy : ', accuracy_score(y_test, pred))
print('MultinomialNB Precision : ', precision_score(y_test, pred))
print('MultinomialNB Recall : ', recall_score(y_test, pred))
print('MultinomialNB F1_Score : ', f1_score(y_test, pred))

print('===================================================')
print(classification_report(y_test, pred))
 
fpr, tpr, thresholds = roc_curve(y_test, prob)
roc_auc = auc(fpr, tpr)
print('AUC:', roc_auc)

In [ ]:
# Import package
import seaborn as sns

f, ax = plt.subplots(figsize=(10,8))
sns.heatmap(confusion_matrix(y_test, pred), annot=True, fmt=".0f", ax=ax)
plt.title("Confusion Matrix")
plt.xlabel("Predictive Value")
plt.ylabel("Actual Value")
plt.show()

In [ ]:
labels = ['POSITIF', 'NEGATIF']
quantity = [df_yTest['Prediksi'].value_counts()]
colors = ['lightskyblue', 'gold']
expl = (0, 0.1)

plt.title('Sentimen Analisis Twitter')
plt.pie(quantity, labels=labels, colors=colors, explode=expl, autopct='%1.1f%%', shadow=True, startangle=90)

plt.axis('equal')
plt.legend(title = 'Sentimen')
plt.show()